In [ ]:
# ============================================================
# CUADERNO 05 — ONE-CLASS SVM
# Semana 10: Ejecutar Experimentos
# Proyecto: Modelos de ML para Detección de Anomalías
#           en el Tráfico y Logs de Entornos Cloud
# ============================================================
# Respaldo metodológico:
# - Bountzis et al. (2025) — OC-SVM kernel RBF
# - Almuhanna & Dardouri (2025) — proporción 70/30
# - Aljuaid & Alshamrani (2024) — métricas evaluación
# Nota: Se usa muestra de 100,000 filas del training set
# para viabilidad computacional del OC-SVM estándar
# ============================================================

In [ ]:
# ── CELDA 1: Instalaciones y configuración ──────────────────
from google.colab import drive
drive.mount('/content/drive')

!pip install memory-profiler psutil -q

import pandas as pd
import numpy as np
import time
import psutil
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.svm import OneClassSVM
from sklearn.model_selection import ParameterGrid
from sklearn.metrics import (f1_score, accuracy_score,
                             precision_score, recall_score,
                             roc_auc_score, confusion_matrix)

ruta_procesados = '/content/drive/MyDrive/IDS2018_procesados'
ruta_resultados = '/content/drive/MyDrive/IDS2018_resultados'
os.makedirs(ruta_resultados, exist_ok=True)

print("✅ Librerías cargadas correctamente")

Mounted at /content/drive
✅ Librerías cargadas correctamente


In [ ]:
# ── CELDA 2: Cargar datos preprocesados ─────────────────────
print("="*60)
print("CARGANDO DATOS PREPROCESADOS")
print("="*60)

df_train_full = pd.read_csv(
    f"{ruta_procesados}/dataset_train.csv")
df_test = pd.read_csv(
    f"{ruta_procesados}/dataset_test.csv")
df_labels = pd.read_csv(
    f"{ruta_procesados}/dataset_test_labels.csv")

# Eliminar columnas no numéricas
df_train_full = df_train_full.select_dtypes(
    include=[np.number])
df_test = df_test.select_dtypes(include=[np.number])

y_test_labels = df_labels.iloc[:, 0]
y_test_binary = (y_test_labels != 'Benign').astype(int)

n_train_70 = int(len(df_train_full) * 0.70)
N_MUESTRA_SVM = 100000

print(f"Training disponible  : {len(df_train_full):,} filas")
print(f"Training 70%         : {n_train_70:,} filas")
print(f"Muestra SVM          : {N_MUESTRA_SVM:,} filas")
print(f"Test set             : {len(df_test):,} filas")
print(f"Features             : {df_train_full.shape[1]}")
print(f"Ataques en test      : {y_test_binary.sum():,}")

CARGANDO DATOS PREPROCESADOS
Training disponible  : 3,805,770 filas
Training 70%         : 2,664,039 filas
Muestra SVM          : 100,000 filas
Test set             : 4,967,041 filas
Features             : 13
Ataques en test      : 1,161,271


In [ ]:
# ── CELDA 3: Grid Search para hiperparámetros ───────────────
# Respaldo: Ness et al. (2025), Samriya et al. (2024)
# Kernel RBF — Bountzis et al. (2025)
# Se usa semilla 42 para la búsqueda de hiperparámetros
print("="*60)
print("GRID SEARCH — HIPERPARAMETROS OC-SVM")
print("Kernel: RBF (Bountzis et al., 2025)")
print("="*60)

np.random.seed(42)
idx_gs = np.random.choice(
    len(df_train_full),
    size=min(30000, len(df_train_full)),
    replace=False
)
X_gs = df_train_full.iloc[idx_gs].values
X_test_gs = df_test.values
y_test_gs = y_test_binary.values

param_grid = {
    'nu': [0.05, 0.1, 0.2],
    'gamma': ['scale', 'auto', 0.01]
}

mejor_f1 = -1
mejores_params = {'nu': 0.1, 'gamma': 'scale'}

for params in ParameterGrid(param_grid):
    try:
        modelo_gs = OneClassSVM(
            kernel='rbf',
            nu=params['nu'],
            gamma=params['gamma']
        )
        modelo_gs.fit(X_gs)
        y_pred_gs = (
            modelo_gs.predict(X_test_gs) == -1
        ).astype(int)
        f1_gs = f1_score(
            y_test_gs, y_pred_gs, zero_division=0)
        print(f"  nu={params['nu']} gamma={params['gamma']}"
              f" → F1={f1_gs:.4f}")
        if f1_gs > mejor_f1:
            mejor_f1 = f1_gs
            mejores_params = params
    except Exception as e:
        print(f"  nu={params['nu']} gamma={params['gamma']}"
              f" → ERROR: {e}")

print(f"\n✅ Mejores hiperparametros: {mejores_params}")
print(f"✅ Mejor F1 en Grid Search: {mejor_f1:.4f}")

GRID SEARCH — HIPERPARAMETROS OC-SVM
Kernel: RBF (Bountzis et al., 2025)
  nu=0.05 gamma=scale → F1=0.2275
  nu=0.1 gamma=scale → F1=0.2041
  nu=0.2 gamma=scale → F1=0.1760
  nu=0.05 gamma=auto → F1=0.0100
  nu=0.1 gamma=auto → F1=0.0097
  nu=0.2 gamma=auto → F1=0.1647
  nu=0.05 gamma=0.01 → F1=0.0097
  nu=0.1 gamma=0.01 → F1=0.0092
  nu=0.2 gamma=0.01 → F1=0.1551

✅ Mejores hiperparametros: {'gamma': 'scale', 'nu': 0.05}
✅ Mejor F1 en Grid Search: 0.2275


In [ ]:
# ── CELDA 4 CORREGIDA: One-Class SVM 5 repeticiones ─────────
import time
import psutil
import os
import numpy as np
import pandas as pd
from sklearn.svm import OneClassSVM
from sklearn.metrics import (f1_score, accuracy_score,
                             precision_score, recall_score,
                             roc_auc_score, confusion_matrix)

# Hiperparámetros óptimos del Grid Search
mejores_params = {'nu': 0.05, 'gamma': 'scale'}
N_MUESTRA_SVM = 50000

print("="*60)
print("ONE-CLASS SVM — 5 REPETICIONES")
print("="*60)
print(f"Hiperparametros optimos:")
print(f"  kernel : rbf")
print(f"  nu     : {mejores_params['nu']}")
print(f"  gamma  : {mejores_params['gamma']}")
print(f"  Muestra training: {N_MUESTRA_SVM:,} filas")
print("="*60)

resultados_svm = []
semillas = [42, 123, 456, 789, 1024]

X_test = df_test.select_dtypes(
    include=[np.number]).values.astype(np.float32)

for i, semilla in enumerate(semillas):
    print(f"\nRepeticion {i+1}/5 (semilla={semilla})...")
    np.random.seed(semilla)

    # Muestra representativa para training
    idx_train = np.random.choice(
        len(df_train_full),
        size=min(N_MUESTRA_SVM, int(len(df_train_full)*0.70)),
        replace=False
    )
    X_train = df_train_full.select_dtypes(
        include=[np.number]).iloc[idx_train].values.astype(
        np.float32)

    # Medir recursos
    proceso = psutil.Process(os.getpid())
    mem_antes = proceso.memory_info().rss / 1024 / 1024

    # Entrenar OC-SVM
    t_inicio = time.time()
    modelo = OneClassSVM(
        kernel='rbf',
        nu=mejores_params['nu'],
        gamma=mejores_params['gamma']
    )
    modelo.fit(X_train)
    t_fin_entren = time.time()
    tiempo_entrenamiento = t_fin_entren - t_inicio

    # Velocidad de inferencia
    t_inicio_inf = time.time()
    y_pred_raw = modelo.predict(X_test)
    t_fin_inf = time.time()
    tiempo_inferencia = t_fin_inf - t_inicio_inf

    # Recursos después
    mem_despues = proceso.memory_info().rss / 1024 / 1024
    cpu_uso = psutil.cpu_percent(interval=1)

    # OC-SVM: -1=anomalía, 1=normal → 0/1
    y_pred = (y_pred_raw == -1).astype(int)

    # Anomaly scores
    decision_scores = -modelo.decision_function(X_test)

    # Métricas VD
    acc = accuracy_score(y_test_binary, y_pred)
    f1 = f1_score(
        y_test_binary, y_pred, zero_division=0)
    prec = precision_score(
        y_test_binary, y_pred, zero_division=0)
    rec = recall_score(
        y_test_binary, y_pred, zero_division=0)

    cm = confusion_matrix(y_test_binary, y_pred)
    tn, fp, fn, tp = cm.ravel()
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0

    try:
        auc = roc_auc_score(y_test_binary, decision_scores)
    except:
        auc = 0.0

    # Métricas VI
    vel_inferencia = len(X_test) / tiempo_inferencia
    uso_memoria = mem_despues - mem_antes

    resultados_svm.append({
        "Repeticion": i + 1,
        "Semilla": semilla,
        # VI
        "Tiempo_entrenamiento_s": round(
            tiempo_entrenamiento, 4),
        "Velocidad_inferencia_mps": round(
            vel_inferencia, 2),
        "Uso_memoria_MB": round(uso_memoria, 2),
        "Uso_CPU_pct": round(cpu_uso, 2),
        # VD
        "Accuracy": round(acc, 4),
        "Precision": round(prec, 4),
        "Recall": round(rec, 4),
        "F1_Score": round(f1, 4),
        "FPR": round(fpr, 4),
        "AUC_ROC": round(auc, 4),
        "Score_medio_normal": round(
            decision_scores[y_test_binary==0].mean(), 4),
        "Score_medio_ataque": round(
            decision_scores[y_test_binary==1].mean(), 4),
    })

    print(f"  ✅ Tiempo entren.   : {tiempo_entrenamiento:.2f} s")
    print(f"  ✅ Vel. inferencia  : {vel_inferencia:,.0f} muestras/s")
    print(f"  ✅ Uso memoria      : {uso_memoria:.2f} MB")
    print(f"  ✅ F1-Score         : {f1:.4f}")
    print(f"  ✅ AUC-ROC          : {auc:.4f}")

print("\n✅ 5 repeticiones completadas")

ONE-CLASS SVM — 5 REPETICIONES
Hiperparametros optimos:
  kernel : rbf
  nu     : 0.05
  gamma  : scale
  Muestra training: 50,000 filas

Repeticion 1/5 (semilla=42)...
  ✅ Tiempo entren.   : 10.60 s
  ✅ Vel. inferencia  : 11,246 muestras/s
  ✅ Uso memoria      : 44.14 MB
  ✅ F1-Score         : 0.2212
  ✅ AUC-ROC          : 0.2278

Repeticion 2/5 (semilla=123)...
  ✅ Tiempo entren.   : 9.59 s
  ✅ Vel. inferencia  : 11,298 muestras/s
  ✅ Uso memoria      : 0.13 MB
  ✅ F1-Score         : 0.2164
  ✅ AUC-ROC          : 0.2273

Repeticion 3/5 (semilla=456)...
  ✅ Tiempo entren.   : 9.72 s
  ✅ Vel. inferencia  : 11,292 muestras/s
  ✅ Uso memoria      : 0.89 MB
  ✅ F1-Score         : 0.2340
  ✅ AUC-ROC          : 0.2271

Repeticion 4/5 (semilla=789)...
  ✅ Tiempo entren.   : 10.02 s
  ✅ Vel. inferencia  : 11,177 muestras/s
  ✅ Uso memoria      : 0.13 MB
  ✅ F1-Score         : 0.2333
  ✅ AUC-ROC          : 0.2275

Repeticion 5/5 (semilla=1024)...
  ✅ Tiempo entren.   : 10.22 s
  ✅ Vel. inferen

In [ ]:
# ── CELDA 5: Resultados y estadísticas ──────────────────────
print("="*60)
print("RESULTADOS — ONE-CLASS SVM")
print("="*60)

df_resultados = pd.DataFrame(resultados_svm)

metricas = [
    "Tiempo_entrenamiento_s",
    "Velocidad_inferencia_mps",
    "Uso_memoria_MB",
    "Uso_CPU_pct",
    "Accuracy",
    "Precision",
    "Recall",
    "F1_Score",
    "FPR",
    "AUC_ROC"
]

print("\nResultados por repeticion:")
print(df_resultados[
    ["Repeticion"] + metricas
].to_string(index=False))

print("\n" + "="*60)
print("MEDIA ± DESVIACION ESTANDAR")
print("="*60)

resumen = {}
for m in metricas:
    media = df_resultados[m].mean()
    std = df_resultados[m].std()
    resumen[m] = f"{media:.4f} ± {std:.4f}"
    print(f"{m:35s}: {media:.4f} ± {std:.4f}")

RESULTADOS — ONE-CLASS SVM

Resultados por repeticion:
 Repeticion  Tiempo_entrenamiento_s  Velocidad_inferencia_mps  Uso_memoria_MB  Uso_CPU_pct  Accuracy  Precision  Recall  F1_Score    FPR  AUC_ROC
          1                 10.5999                  11245.57           44.14          6.0    0.7572     0.4421  0.1475    0.2212 0.0568   0.2278
          2                  9.5853                  11297.53            0.13          1.1    0.7499     0.4047  0.1477    0.2164 0.0663   0.2273
          3                  9.7246                  11292.37            0.89          0.5    0.7742     0.5653  0.1475    0.2340 0.0346   0.2271
          4                 10.0226                  11176.63            0.13          0.4    0.7733     0.5570  0.1475    0.2333 0.0358   0.2275
          5                 10.2172                  11275.47          -35.64          0.6    0.7576     0.4445  0.1477    0.2217 0.0563   0.2269

MEDIA ± DESVIACION ESTANDAR
Tiempo_entrenamiento_s             : 10.

In [ ]:
# ── CELDA 6: Guardar resultados ──────────────────────────────
print("="*60)
print("GUARDANDO RESULTADOS")
print("="*60)

ruta_csv = f"{ruta_resultados}/OCSVM_resultados_detallados.csv"
df_resultados.to_csv(ruta_csv, index=False)
print(f"✅ Resultados detallados: {ruta_csv}")

resumen_df = pd.DataFrame({
    "Metrica": list(resumen.keys()),
    "Media_±_Std": list(resumen.values())
})
ruta_resumen = f"{ruta_resultados}/OCSVM_resumen.csv"
resumen_df.to_csv(ruta_resumen, index=False)
print(f"✅ Resumen guardado     : {ruta_resumen}")

print("\n" + "="*60)
print("ONE-CLASS SVM COMPLETADO")
print("="*60)
print(f"Archivos guardados en  : {ruta_resultados}")
print(f"nu optimo usado        : {mejores_params['nu']}")
print(f"gamma optimo usado     : {mejores_params['gamma']}")
print(f"Muestra training usada : {N_MUESTRA_SVM:,} filas")
print("\nSiguiente paso: Cuaderno 06 — LSTM Autoencoder")

GUARDANDO RESULTADOS
✅ Resultados detallados: /content/drive/MyDrive/IDS2018_resultados/OCSVM_resultados_detallados.csv
✅ Resumen guardado     : /content/drive/MyDrive/IDS2018_resultados/OCSVM_resumen.csv

ONE-CLASS SVM COMPLETADO
Archivos guardados en  : /content/drive/MyDrive/IDS2018_resultados
nu optimo usado        : 0.05
gamma optimo usado     : scale
Muestra training usada : 50,000 filas

Siguiente paso: Cuaderno 06 — LSTM Autoencoder
